# Step 16.4 — Reproducibility：从 Seed 到可继续的 RNG State

16.3 已经给出 `get_batch(data, batch_size, block_size, rng)`：输入一条选定 split，返回 `X/Y:(B,T)` 和长度 B 的 `starts`。本节保留这个接口，回答三个问题：

1. 怎样从相同起点重新得到一段采样过程？
2. 在什么条件下，可以预期相同的 starts 与 X/Y？
3. 怎样保留“已经走到哪里”，为之后继续运行做好准备？

这里的 **Reproducibility（可复现性）** 先聚焦数据采样。模型训练是否逐位一致，是需要更多条件的后续问题。

本母版是准备好的课程代码与讲解，所有 code cells 均未执行。本轮没有读取或下载语料，没有采样、复现实验、测试、训练或用户 work copy 操作，也不填写练习答案。

状态沿用：16.1=1/2、16.2=1/2；16.3 母版及完整回复核心已由用户独立核验 PASS，实际修正仍为 0/2。本节初版实际修正 **0/2**。材料状态不代表学员掌握或运行验证。**本轮停在 16.4，不进入 16.5。**


## 0. 新 Work Copy 的依赖：保留算法，重新建立本节对象

新 Notebook 可以有自己的 kernel。本母版自带 16.3 的 imports，以及 `find_project_root`、`load_cached_character_corpus`、`split_train_validation`、`get_batch` 的原有定义，不依赖旧 Notebook 的变量，也不 `%run` 整份旧课。

四个 helper 的职责分别是：定位项目、读取既有本地缓存并重建字符编码、连续 90/10 切分、在指定 split 内采样。下面只是导入与定义，不调用读取、切分或采样。

未来在 work copy 中使用时，选择项目 `.venv` 的 Python 3.12 kernel。默认缓存是仓库根目录下的 `data/tinyshakespeare/input.txt`；如果 16.1 当时把缓存放在其他目录，将路径 cell 的 `CACHE_PATH` 改为那个**既有文件的绝对路径**。

`Path.cwd()` 是 kernel 当前目录，`resolve()` 显示绝对路径。路径 cell 只定位和显示文件位置；下一份 bootstrap usage 才读取文本并恢复 `characters/stoi/itos/token_stream/train_tokens/val_tokens/split_index`。缺少缓存会明确报错，之后实际使用时回到 16.1 完成 acquisition，或修正路径；本节没有下载逻辑。

**恢复数据与函数定义，不等于追回旧随机序列。** 16.3 使用没有显式 seed 的 `random.Random()`；若当时没有保存 state，不能凭本节新设的 seed 找回那次采样位置。本节从明确的 seed 开始，并学习此后应保留什么。


In [ ]:
from pathlib import Path
import random
import mlx.core as mx


def find_project_root(start: Path) -> Path:
    """Find this course repository from its root or a child directory."""
    current = Path(start).resolve()
    for candidate in [current] + list(current.parents):
        project_file = candidate / 'pyproject.toml'
        previous_lesson = (
            candidate / 'notebooks' /
            '68_step16_2_train_validation_split.ipynb'
        )
        if project_file.is_file() and previous_lesson.is_file():
            return candidate
    raise FileNotFoundError(
        'Course repository not found above the kernel working directory. '
        'Open the mlx-minigpt-from-scratch folder in VS Code first.'
    )


def load_cached_character_corpus(cache_path: Path):
    """Reuse the existing local cache; never download data."""
    cache_path = Path(cache_path)
    if not cache_path.is_file():
        raise FileNotFoundError(
            f'Tiny Shakespeare cache not found at {cache_path.resolve()}. '
            'Point CACHE_PATH at the existing Step 16.1 input.txt, '
            'or complete acquisition in the Step 16.1 work copy first.'
        )

    text = cache_path.read_text(encoding='utf-8')
    characters = sorted(set(text))
    stoi = {ch: i for i, ch in enumerate(characters)}
    itos = {i: ch for i, ch in enumerate(characters)}
    encoded_ids = [stoi[ch] for ch in text]
    token_stream = mx.array(encoded_ids, dtype=mx.int32)
    return text, characters, stoi, itos, token_stream


def split_train_validation(token_stream, train_fraction: float = 0.90):
    """Reuse Step 16.2's contiguous, non-empty train/validation split."""
    if len(token_stream.shape) != 1:
        raise ValueError('token_stream must have shape (N,)')
    N = token_stream.shape[0]
    if N < 2:
        raise ValueError('token_stream must contain at least 2 tokens')
    if not 0.0 < train_fraction < 1.0:
        raise ValueError('train_fraction must satisfy 0 < train_fraction < 1')

    split_index = int(N * train_fraction)
    if split_index <= 0 or split_index >= N:
        raise ValueError('split would make train or validation data empty')

    train_tokens = token_stream[:split_index]
    val_tokens = token_stream[split_index:]
    return train_tokens, val_tokens, split_index


# Definitions only. No cache read, split, or sampling is performed here.


def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


# Definition only. No batch is sampled in this cell.


### 路径与 Bootstrap Usage：只在未来的 Work Copy 中按需使用

先看 `selected cache` 是否指向 16.1 的那份文件，再考虑 bootstrap。读取与切分仍沿用前节：字符词表由 `sorted(set(text))` 建立，token dtype 为 `mx.int32`，两条 split 都是一维。

如果当前只阅读本节 seed/state 的推理，可以直接阅读后续文字，无需访问语料。下面两份 usage 只是备好，不是本轮执行记录。


In [ ]:
# USER-RUN PATH CELL — prepared only, NOT executed in the master.

PROJECT_ROOT = find_project_root(Path.cwd())
CACHE_PATH = PROJECT_ROOT / 'data' / 'tinyshakespeare' / 'input.txt'

# If the existing Step 16.1 cache is elsewhere, replace CACHE_PATH here:
# CACHE_PATH = Path('/absolute/path/to/the/existing/input.txt')

print('kernel cwd:', Path.cwd())
print('project root:', PROJECT_ROOT)
print('selected cache:', CACHE_PATH.resolve())
print('cache is a file:', CACHE_PATH.is_file())


In [ ]:
# USER-RUN BOOTSTRAP CELL — prepared only, NOT executed in the master.

text, characters, stoi, itos, token_stream = load_cached_character_corpus(
    CACHE_PATH,
)
train_tokens, val_tokens, split_index = split_train_validation(
    token_stream,
    train_fraction=0.90,
)

print('full stream:', token_stream.shape, token_stream.dtype)
print('train stream:', train_tokens.shape)
print('validation stream:', val_tokens.shape)
print('split index:', split_index)
print('vocabulary size:', len(characters))

# This usage cell only reads the existing local cache and rebuilds the split.
# It does not access the network or run a model.


## 1. Seed 是起点，State 是走到哪里

**PRNG（Pseudo-Random Number Generator，伪随机数生成器）** 是依据内部状态产生数值、并更新自身状态的算法。它适合让我们在控制条件后重新播放相同过程。

- **Random seed（随机种子）**：初始化这个过程的输入，本节使用普通整数。
- **RNG state（随机数生成器状态）**：生成器当前的完整内部状态。它随采样调用推进，不能用“最初的 seed”代替。
- **Restart / replay（从头重启 / 重放）**：重新建立相同的初始状态。
- **Resume（从中途继续）**：恢复停下时的状态，接着下一次调用。

可以把 seed 想成一本固定版本的书，state 想成阅读位置的书签。重新按 seed 开始会回到书头；保存书签才能回到刚才的位置。这个比喻只帮助理解，state 实际上是算法状态，不是简单页码或一个 batch 计数。

### 用本节已有的 Sampler 表达状态变化

令 $S_k$ 表示某一路 RNG 在**第 k 个 batch 已经抽完、下一批开始前**的状态，$S_0$ 是尚未抽取 batch 时的状态。固定 sampler 后：

$$
S_0=\operatorname{Init}(\text{seed}),
$$

$$
(\text{starts}_k,S_{k+1})
=G(S_k;B,N_{split}-T).
$$

这里 k 从 0 开始编号；`starts_0` 对应第一批。$G$ 就是 16.3 的 B 次 `randrange(0, N_split-T)` 调用。每个起点满足 $0\le s<N_{split}-T$。这里按一次完整 batch 描述状态变化，不假定一次 `randrange` 只消耗一次底层随机位生成。

`random.Random(2026)` 创建一个带有该初始 seed 的新对象；对已有对象调用 `rng.seed(2026)` 会重新初始化它。反复使用同一个对象，会继续推进状态；在每批之前重新创建同 seed 的对象，会反复从同一个起点开始。

### 小例子：重放第一批，还是继续第二批？

仅为手算，设数据为 `[0,1,2,3,4,5,6,7]`，`B=2,T=3`，并**人为假设**某一条随机调用轨迹的前两批起点为 `[1,4]`、`[0,3]`。这些不是 seed=2026 的实测输出，也不声称任何具体 seed 必然产生它们。

| 时间位置 | starts | X 两行 | Y 两行 |
|---|---|---|---|
| 第一批 | `[1,4]` | `[1,2,3]`；`[4,5,6]` | `[2,3,4]`；`[5,6,7]` |
| 第一批后继续 | `[0,3]` | `[0,1,2]`；`[3,4,5]` | `[1,2,3]`；`[4,5,6]` |

每一批的 X/Y 都是 `(2,3)`。在相同条件下，从原 seed 重启会重放第一批；恢复第一批之后的 state 会继续第二批。真实采样允许起点或 batch 偶然重复，因此“状态继续推进”不保证每次返回不同数值。

Python API 依据：[Random / seed / getstate / setstate](https://docs.python.org/3.12/library/random.html)。


## 2. 分开管理 Train、Validation 与 MLX 的随机来源

16.3 的 `rng` 参数使随机来源可以明确传入。假如 train 和 validation 共用同一个对象，增加一次 validation 采样会推进这个共享对象，后面的 training 就不再从原状态取样。

本节创建两份独立状态：

| 对象 / API | 本节职责 |
|---|---|
| `train_rng = random.Random(TRAIN_SEED)` | 只供 training split 的 sampler 使用 |
| `val_rng = random.Random(VAL_SEED)` | 只供 validation split 的 sampler 使用 |
| `mx.random.seed(MLX_SEED)` | 设置 MLX 默认随机源的初始 seed，供以后 MLX 随机操作使用 |

这里“独立”指两个对象的状态可分别推进，不是在证明它们的统计独立性。不同 seed 也不保证两个 batch 永不相同。单纯执行 `val_rng = train_rng` 只是同一个对象的另一个名字，不能实现隔离。

Python 模块级 `random.seed(...)` 控制模块自带的默认生成器，不会替上面显式创建的两个对象设置 seed。`mx.random.seed(...)` 则属于另一套 MLX RNG，也不会改变 Python sampler 的状态。

本 sampler 的随机选择只来自 `rng.randrange(...)`；`mx.array`、切片和 `mx.stack` 负责组织已经决定的整数数据。将来构造 MiniGPT 时，MLX 的随机参数初始化需要在**构造模型之前**设置 MLX seed。本节不构造模型，也不生成 MLX 随机张量。

### 初始化 Cell 的操作含义

下面用三个容易记录的整数作为 seed；数值没有特殊训练优势。`train_batches_sampled`、`val_batches_sampled` 记录各自**已经完成的采样调用数**，起始为 0；它们不是训练 update step，也不是 RNG state。

这份 cell 用于从头开始。重新运行它会重置两路采样状态及计数；继续采样时不要每次先运行它。MLX 的 seed 也只在需要重新开始 MLX 随机过程时重设。

MLX API 依据：[random.seed](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.random.seed.html)，并已按项目固定的 [v0.32.2 源码](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/random.cpp) 核对。


In [ ]:
# USER-RUN FRESH-START SETUP — prepared only, NOT executed in the master.
# Re-running this cell restarts the sampling streams; it is not a resume.

B = 2
block_size = 8
TRAIN_SEED = 2026
VAL_SEED = 2027
MLX_SEED = 2028

train_rng = random.Random(TRAIN_SEED)
val_rng = random.Random(VAL_SEED)
mx.random.seed(MLX_SEED)

train_batches_sampled = 0
val_batches_sampled = 0

# No batch is sampled and no model is created here.


## 3. Deterministic Sampling Expectations：到底可以预期什么一致？

**Deterministic（确定性的）** 在这里表示：给定相同起始状态和相同操作序列，结果由这些条件确定。不是“随机数固定不动”，也不是“整个训练在任何机器上都一样”。

先区分两层：

1. **相同 starts**：需要兼容且固定的 Python 实现/版本、相同 sampler 代码、相同 RNG 起始状态，以及同样的 `randrange` 参数和调用次序。本节对应同样的 B、T、`N_split`，且中间没有额外消费该 RNG。
2. **相同 X/Y**：在 starts 相同之外，还需要同样的整数 token stream、字符到 ID 的映射和 split 内容/顺序。

依然有：

$$
X[b,t]=data[starts[b]+t],\qquad
Y[b,t]=data[starts[b]+t+1].
$$

所以同样的起点配上不同数据，并不能推出同样的 X/Y。相同文件名、同样长度也不能证明文件内容没变。

| 情况 | 应有的预期 |
|---|---|
| 两个新对象使用同 seed，并按相同条件调用 | 可以重放相同的对应采样序列 |
| 对一个对象连续调用 | 状态继续推进；不承诺每次结果不同 |
| 更改 B、T、split 长度或加入额外随机调用 | 复现条件改变，不能承诺原来的后续 batches |
| 只对独立的 `val_rng` 增加调用 | 不消费 `train_rng`；会改变 validation 自己之后的位置 |
| 只设置 MLX seed | 不能控制当前 Python sampler 的 starts |
| 更换 Python / MLX 版本、设备或执行配置 | 不由 seed 推出完整训练逐位一致 |

Python 的跨版本兼容保证有特定范围，不能直接推广为所有版本的 `randrange` 都不变。项目仍使用 Python 3.12 与 `mlx==0.32.2`；将来应记录实际软件版本与代码版本，而不是只写一个 seed。[官方复现说明](https://docs.python.org/3.12/library/random.html#notes-on-reproducibility)

### 准备一次连续使用：保持 16.3 的 Shape 契约

下面的 cell 从两路 RNG 的**当前位置**各取一批。`+= 1` 在相应调用成功返回后，把该路已完成的采样计数加一；它不推进另一条 RNG。

所有 X/Y 的预期 shape 是 `(B,T)=(2,8)`，dtype 沿用 `mx.int32`。两个 starts 都是长度 2 的 Python list，属于各自 split 的局部下标。代码只准备打印信息，不包含实际运行结果或断言测试。


In [ ]:
# USER-RUN SAMPLING — prepared only, NOT executed in the master.
# Re-run this cell to continue; do not re-run fresh-start setup first.

X_train, Y_train, train_starts = get_batch(
    train_tokens, B, block_size, train_rng,
)
train_batches_sampled += 1

X_val, Y_val, val_starts = get_batch(
    val_tokens, B, block_size, val_rng,
)
val_batches_sampled += 1

print('train split-local starts:', train_starts)
print('validation split-local starts:', val_starts)
print('train shapes:', X_train.shape, Y_train.shape)
print('validation shapes:', X_val.shape, Y_val.shape)
print('completed sampling calls:', train_batches_sampled, val_batches_sampled)

# Expected by construction, not measured here:
# X/Y: (2,8), mx.int32; each starts list: length 2.


## 4. 从中途继续：先保存 State，再恢复到新对象

现在才引入两个新 API：

- `rng.getstate()`：取得该对象**当前**的内部状态快照，不抽取新的起点。
- `rng.setstate(saved_state)`：把先前 `getstate()` 返回的状态装入这个对象；之后的随机调用从该位置继续。

这里把返回值当作 **opaque state（不展开内部结构的状态对象）**，完整保存并交还 API，不手改里面的数。`saved_rng = train_rng` 只是保留同一个会继续变化的对象引用，不能代替 `getstate()`。

### 选择明确的边界：上一批已抽完，下一批尚未抽取

本节快照保存两路 RNG state、B/T 和两个采样计数。Python `dict` 用 `key: value` 保存对应信息，之后用 `snapshot['key']` 取回。

下方 cell 应位于采样 cell **完整返回后**，中间不插入其他 RNG 调用。快照描述的是下一次调用前的位置，恢复后要取的是**下一批**，不是重新返回刚才的 X/Y。前面那批的数据并不包含在 state 内。

本例只做 **in-memory snapshot（内存快照）**：保持同一个 kernel、同样的 `train_tokens/val_tokens`、词表、split、sampler 代码和软件环境不变，演示把状态交给新对象。快照没有把这些数据/定义打包进去。字典留在内存里，kernel 终止后会消失；跨进程持久化留给 Step 20。


In [ ]:
# USER-RUN SNAPSHOT — prepared only, NOT executed in the master.
# Place this after the preceding sampling cell has finished.

sampling_snapshot = {
    'train_rng_state': train_rng.getstate(),
    'val_rng_state': val_rng.getstate(),
    'batch_size': B,
    'block_size': block_size,
    'train_batches_sampled': train_batches_sampled,
    'val_batches_sampled': val_batches_sampled,
}

# This captures Python sampling state only. It writes no file.
# Keep the same data, vocabulary, split, sampler, and runtime in memory.


### 恢复时先还原状态，再取下一批

先创建两个新的 `random.Random(0)` 对象，再分别调用 `setstate(...)`。这里的 0 只是创建对象时的占位初始 seed，马上会被保存的完整 state 覆盖；它不是“恢复必须用的 seed”。

同时还原 B/T 和采样计数，使之后的调用语义与保存时一致。下面没有重设 MLX seed，因为这个示例只交接 Python sampler 状态；它没有保存或恢复 MLX RNG、模型和优化器。

恢复 cell 本身不采样。后续 usage 才各取下一批；如果回头再次执行恢复 cell，会再次回到同一个快照位置，随后重放那里的下一批。真正想持续往后走时，只继续采样，不反复恢复。


In [ ]:
# USER-RUN IN-MEMORY RESTORE — prepared only, NOT executed in the master.
# Requires sampling_snapshot and the unchanged data/definitions in this kernel.

train_rng = random.Random(0)
val_rng = random.Random(0)
train_rng.setstate(sampling_snapshot['train_rng_state'])
val_rng.setstate(sampling_snapshot['val_rng_state'])

B = sampling_snapshot['batch_size']
block_size = sampling_snapshot['block_size']
train_batches_sampled = sampling_snapshot['train_batches_sampled']
val_batches_sampled = sampling_snapshot['val_batches_sampled']

# No sampling, data read, MLX reseeding, or file access occurs in this cell.


In [ ]:
# USER-RUN NEXT BATCH AFTER RESTORE — prepared only, NOT executed in the master.

X_train_next, Y_train_next, train_starts_next = get_batch(
    train_tokens, B, block_size, train_rng,
)
train_batches_sampled += 1

X_val_next, Y_val_next, val_starts_next = get_batch(
    val_tokens, B, block_size, val_rng,
)
val_batches_sampled += 1

print('next train starts:', train_starts_next)
print('next validation starts:', val_starts_next)
print('next train shapes:', X_train_next.shape, Y_train_next.shape)
print('next validation shapes:', X_val_next.shape, Y_val_next.shape)
print('completed sampling calls:', train_batches_sampled, val_batches_sampled)

# These are the batches after the saved boundary, not the batches before it.
# No expected random values or runtime comparison results are supplied.


## 5. 真正恢复继续还需要什么？先建立状态清单

本节内存示例的前提是数据与环境仍在原地。未来进程退出后还要继续，必须把这些前提变成能恢复、能核对的记录。

| 状态 / 信息 | 为什么需要 |
|---|---|
| Python train / validation RNG 的完整 state | 接着两路各自的随机位置；原始 seeds 用于说明起点，不能代替当前位置 |
| 准确的数据内容及顺序、`characters/stoi/itos`、split 边界 | 确保同一 start 对应同一文本与 token IDs；仅文件路径、长度不足以证明相同数据 |
| B、T、sampler 代码和各路已完成的采样计数 | 确认调用规则和进度；计数本身不能直接恢复 RNG state |
| Python/MLX 版本、代码提交、设备与 dtype 等执行配置 | 说明复现所处的环境，避免把状态用于不同语义的程序 |
| 未来模型参数与架构配置 | 保留模型已经学到的数值，并按正确结构重建；本节不创建模型 |
| 未来 optimizer state 与已完成的 training update step | optimizer state 是优化器随训练积累、影响下一次更新的内部量；update step 决定下一次更新位置，与采样计数不同 |
| 后续仍会使用的 MLX RNG state / 其他随机来源状态 | 继续 MLX 侧的随机操作；只保存 Python state 不会覆盖它们 |

数据身份可以由固定数据版本及内容指纹记录；这里只说明应保存的含义，不实现哈希或文件打包。MLX RNG 的真实保存与恢复要按项目版本采用合适接口，本节不将 `mx.random.state` 当作可随意赋值的普通列表。

**Checkpoint（可持久化的恢复检查点）** 要在同一个逻辑边界记录一致的状态。未来训练中，“完成某次参数更新之后、尚未抽下一批”是容易说明的边界。如果已经抽了一批但尚未训练它，却只保存抽取后的 RNG 状态，恢复后直接抽下一批就会跳过这份待处理数据。本节没有参数更新，因此这里只把采样后的边界说清楚，不把这个字典称为完整训练 checkpoint。

当前算法随机取窗口，没有“顺序读完整份数据的游标”。本节不提前实现 16.5 的工业输入管线，也不实现 Step 20 的文件保存、完整训练恢复或恢复验证。


## Work Copy 的可操作顺序与本节终点

之后需要实际使用时：

1. `git pull` 后，将本节母版安全复制为 `notebooks/work/` 下的新文件名，只打开副本，选择项目 Python 3.12 / `.venv` kernel。
2. 运行 imports / helper / sampler 的定义 cell。
3. 运行路径 cell；必要时将 `CACHE_PATH` 改为 16.1 既有缓存的绝对路径，再运行 bootstrap，取得两条 split 和字符映射。定义与 bootstrap 都在本母版中。
4. 读懂 seed 与 state 后，运行一次 fresh-start setup，建立两路采样 RNG、MLX seed 和采样计数。
5. 使用 sampling cell 从当前位置取 batch；不在每个 batch 前重设 seed。
6. 需要学习状态交接时，在采样完成的边界取得 snapshot，再运行 in-memory restore，之后才取下一批。这个示例要求相同 kernel 内的数据与定义保持不变。

如果 kernel 已退出，本节的内存 snapshot 就不存在；本母版仍能从既有缓存和明确 seeds 重新开始，但不能假装从失去的中途位置精确继续。真正持久化方案按 roadmap 留给 Step 20。

以上是未来 work copy 的操作说明，本轮没有代替用户执行。这里没有需要代填的练习答案，也没有将预期 shape 或随机关系写成运行测量结果。

本节完成的材料接口是：**明确 seed 与 RNG 所有者 → 说明采样复现条件 → 在明确边界捕获/恢复 Python RNG state → 列出完整恢复所需信息**。

**本轮止于 Step 16.4，不进入 16.5。** 实际修正累计：16.1=1/2、16.2=1/2、16.3=0/2、16.4 初版=0/2。后续按用户请求逐小节推进，至大 Step 20 全部完成后停止；本节发布不代表学员掌握或运行验证。


## 参考与衔接

- 前置：[Step 16.3 母版](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/notebooks/69_step16_3_batch_sampler.ipynb)、[Step 16 文档](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/step-16-tiny-shakespeare-data.md)。
- [Python 3.12 random](https://docs.python.org/3.12/library/random.html)：独立实例、seed、getstate/setstate、randrange 与复现条件。
- [CPython 3.12.0 Random 实现](https://github.com/python/cpython/blob/v3.12.0/Lib/random.py)：state API 与 randrange 内部抽取方式的只读依据。
- [MLX random.seed API](https://ml-explore.github.io/mlx/build/html/python/_autosummary/mlx.core.random.seed.html)、[MLX Random](https://ml-explore.github.io/mlx/build/html/python/random.html)：MLX 随机来源的分工。
- [MLX v0.32.2 random binding](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/random.cpp)：固定项目版本的 API 依据。在线文档可能随最新版本更新，项目仍固定 `mlx==0.32.2`。
- 之后的 16.5 与 Step 20 仍按 [roadmap](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/main/docs/course-roadmap.md) 分节准备；本轮不进入。
